# X5_Q3 — Hipótesis parámetro↔variable + tabla final

Tercer y último notebook de **X5_alt_v2** (ver
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10,
documento rector [`docs/plans/x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)).
TODOs en `docs/tracking/todos.md`, sección X5_alt_v2.

Consume la tabla de estado de `X5_Q1.ipynb`
(`resources/x5_alt/{valor}_tabla_estado_v2.csv`) y el export estructurado de
métricas de `X5_Q2.ipynb` (`resources/x5_alt/{valor}_q2_5_export_metricas.json`)
para traducir la evidencia descriptiva acumulada en hipótesis explícitas
`parámetro X5 ↔ variable` — sin implementarlas todavía como reglas de
trading (eso queda para una etapa posterior de backtesting).

Gate de arranque (`[Q3-1]`): requiere `Q1-EXEC` y `Q2-EXEC` ya satisfechos —
ver `docs/tracking/done.md`.

Mismo input por lista `valores` que `X5_Q1.ipynb`/`X5_Q2.ipynb` (no el
`valor` singular de v1).

In [1]:
valores = ['BTCUSD']  # lista de activos; default solo BTCUSD (ver plan X5_alternativo.md §10.2)


## 0. Preparación

In [2]:
import json
import sys
from pathlib import Path

import pandas as pd

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

print(f"Activos: {valores}")


Activos: ['BTCUSD']


## 1. Carga de la tabla de estado de Q1 y el export de métricas de Q2 (Q3-1)

Lee, por cada activo de `valores`, `{valor}_tabla_estado_v2.csv` (`X5_Q1.ipynb`,
Sección 10) y `{valor}_q2_5_export_metricas.json` (`X5_Q2.ipynb`, Sección 14) —
ninguno de los dos se recalcula acá. Si falta alguno, corrió incompleto el
notebook correspondiente.

In [3]:
def _cargar_tabla_estado(valor):
    path = cfg.CARPETA_X5_ALT / f'{valor}_tabla_estado_v2.csv'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q1.ipynb primero para generarla.")
    return pd.read_csv(path, parse_dates=['DateTime'])


def _cargar_export_metricas(valor):
    path = cfg.CARPETA_X5_ALT / f'{valor}_q2_5_export_metricas.json'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q2.ipynb primero para generarlo.")
    with open(path, encoding='utf-8') as f:
        return json.load(f)


tablas = {valor: _cargar_tabla_estado(valor) for valor in valores}
export_metricas = {valor: _cargar_export_metricas(valor) for valor in valores}

for valor in valores:
    tabla = tablas[valor]
    print(f"{valor}: tabla_estado {tabla.shape[0]} filas x {tabla.shape[1]} columnas — "
          f"{tabla['DateTime'].min()} → {tabla['DateTime'].max()} | "
          f"export_metricas: {len(export_metricas[valor])} variables")


BTCUSD: tabla_estado 41892 filas x 72 columnas — 2022-01-01 03:00:00 → 2026-10-12 18:00:00 | export_metricas: 63 variables


## 2. Alcance de este notebook (Q3-3)

**`X5_Q3.ipynb` no implementa reglas de trading.** Su único producto son
hipótesis explícitas del tipo `parámetro X5 = f(variable(s) de estado del
activo)` (`N`, `A`, `factor_B`, `K`, `N_EXP`, `LAMBDA`), apoyadas en la
evidencia descriptiva de `X5_Q2.ipynb` y en el pre-registro de
[`X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10.5 (`[PRE-1]`).

Quedan explícitamente fuera de este notebook:

- Implementar cualquiera de estas relaciones como código de producción en
  `X0_data_supports.py`/`X1_trading.py`.
- Validar las hipótesis con backtesting — eso es responsabilidad de una
  etapa posterior (puente a `X4_backtester.py` / Fase 3 de `X5_alt`, ver
  TO DO "X5 — estudio posterior de parámetros manipulables y variables
  explicativas" en `docs/tracking/todos.md`).

Una hipótesis con evidencia fuerte en Q2 sigue siendo una hipótesis hasta que
el backtesting la confirme.

## 3. Vista unificada de variables explicativas + precio (Q3-8)

Tabla/export, por activo, con `DateTime`, `Close` y las variables explicativas
**relevantes** — para inspección conjunta (graficar, revisar a ojo), no para
modelar. No recalcula nada: reutiliza el `nivel_evidencia_descriptiva` ya
calculado en `Q2-5`/`Q2-11` dentro de `export_metricas`.

"Relevante" = `nivel_evidencia_descriptiva` en `{alta, media}` contra
`retorno_log_1h` — el target principal. `Close` (nivel) es solo referencial
en toda la síntesis de `Q2-11`: ya se vio ahí que casi ninguna variable
sobrevive el filtro de evidencia contra el nivel de precio, a diferencia del
retorno.

In [4]:
TARGET_PRINCIPAL_Q3 = 'retorno_log_1h'  # Close es solo referencial (ver síntesis Q2-11)
NIVELES_RELEVANTES_Q3_8 = {'alta', 'media'}


def _variables_relevantes(valor):
    metricas = export_metricas[valor]
    return sorted(
        variable for variable, info in metricas.items()
        if info['targets'][TARGET_PRINCIPAL_Q3]['nivel_evidencia_descriptiva'] in NIVELES_RELEVANTES_Q3_8
    )


vistas_unificadas = {}
for valor in valores:
    variables_relevantes = _variables_relevantes(valor)
    vista = tablas[valor][['DateTime', 'Close'] + variables_relevantes].copy()
    vistas_unificadas[valor] = vista
    out_path = cfg.CARPETA_X5_ALT / f'{valor}_q3_8_vista_unificada.csv'
    vista.to_csv(out_path, index=False)
    print(f"{valor}: {len(variables_relevantes)} variables relevantes (nivel {NIVELES_RELEVANTES_Q3_8} "
          f"contra {TARGET_PRINCIPAL_Q3}) -> {out_path.name}")


BTCUSD: 19 variables relevantes (nivel {'alta', 'media'} contra retorno_log_1h) -> BTCUSD_q3_8_vista_unificada.csv


## 4. Hipótesis prioritaria: N = f(drawdown_activo) (Q3-2)

**Qué controla**: `N` (`n_sizes_ejecucion`) es la cantidad de soportes activos
que X1 mantiene por activo. ↑N → más cobertura del rango y entradas más
finas, pero capital más fragmentado. ↓N → posiciones más concentradas,
cobertura más gruesa.

**Hipótesis** (documento rector [`x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)
sección 6, pre-registrada en [`X5_alternativo.md`](../docs/plans/X5_alternativo.md)
§10.5 `[PRE-1]`): drawdown del activo sano (poco profundo/reciente) + volatilidad
baja/normal → `N` alto. Drawdown profundo/no sano + volatilidad alta → `N` bajo.
Nunca drawdown/equity de la cuenta — solo del precio del activo.

**Signo esperado**: relación directa entre la "salud" del drawdown (menos
profundo, más cercano a 0) y `N` — cuanto más profundo/negativo el drawdown,
más bajo `N`.

**Variables candidatas**: profundidad y duración del drawdown en distintas
ventanas (`drawdown_7d`/`30d`/`90d`/`365d`, `drawdown_historico`), tiempo
desde el máximo, velocidad de caída/recuperación, volatilidad corta/larga.

**Evidencia descriptiva disponible** (`Q2-11`, contra `retorno_log_1h` en
BTCUSD): las ventanas cortas de drawdown tienen evidencia **alta** y signo
consistente con la hipótesis — `x3_drawdown_20` (Pearson 0.28), `x3_drawdown_50`
(0.19) y `drawdown_7d` (0.11) correlacionan positivo con el retorno siguiente
(drawdown menos profundo → retorno algo mayor), estables en el tiempo y sin
dependencia de régimen. Las ventanas largas (`drawdown_90d`/`365d`/`historico`)
caen a evidencia **baja** — la señal se debilita cuanto más se alarga la
ventana. Esto es evidencia sobre la relación variable↔retorno futuro, no
sobre el parámetro `N` en sí — no reemplaza el backtesting.

**Rango permitido**: mismo rango de exploración ya usado por X5
(`X5_PARAM_RANGES`, `config.py`, ver [`pars_configurables.md`](../docs/context/pars_configurables.md)) —
`(50, 200)` para BTCUSD/ETHUSD, `(40, 180)` para TSLA/GOOGL/NVDA/AMZN.

**Riesgo económico asociado**: `N` bajo en drawdown profundo reduce la
exposición simultánea (menos soportes = menos entradas abiertas a la vez),
acotando la pérdida potencial agregada por activo, a costa de menor cobertura
si el drawdown resulta ser temporal. `N` alto en drawdown sano maximiza
cobertura/entradas finas, pero fragmenta capital y multiplica las posiciones
simultáneas si el drawdown se profundiza inesperadamente después.

**Cómo validar**: backtesting (`X4_backtester.py`) comparando desempeño con
`N` dinámico `N=f(drawdown_activo)` contra el `N` estático actual (180 fijo),
por activo — el análisis descriptivo de Q2 es insumo para la hipótesis, no
su validación.